# 第4章 動画の「いつ」をつかむ (時間区間の検出)

第3章の座標は「画像のどこ」でした。この章は「動画のいつ」です。Cosmos 3 の Reasoner は、動画の中の出来事を**始まりと終わりの時刻つき**で答えられます。

1. 動画の中の動作を区切って列挙させる (秒で答える)
2. 出来事の年表を作らせる (`mm:ss.ff` で答える)
3. 「この場面はいつ?」と時刻を聞く
4. 「この区間に何が起きた?」と聞く
5. 車載カメラの動画で試す
6. 自分の動画で試す

答えの時刻をグラフにし、その時刻のコマを切り出して、本当に合っているかを目で確かめます。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外。Ampere 以降の GPU と bf16 が前提です)。
Reasoner (文章で答える部分) だけを使うので、GPU メモリは 10GB 前後で足ります。

上から順に実行してください。「モデルを読み込む」セルは初回に 5GB ほどダウンロードします。

In [ ]:
# @title GPU を確かめる
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    print("GPU が見つかりません。Colab ならランタイムのタイプで L4 か A100 を選んでください。")

In [ ]:
# @title インストール (1〜2 分)
!pip install -q -U "transformers>=5.18" accelerate av huggingface_hub

In [ ]:
# @title サンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
# NVIDIA の公式クックブック (cosmos-framework) とモデルカードに置かれているお手本の素材
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title モデルを読み込む (初回は 5GB ほどダウンロード)
import json
import re

from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageDraw, ImageOps
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "nvidia/Cosmos3-Edge"
# Transformers は統合チェックポイントから Reasoner の部分だけを読み込む (生成用の部分は捨てる)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
# @title 質問用の関数を用意する
import json
import re

import av
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from transformers.video_utils import VideoMetadata

VIDEO_EXT = (".mp4", ".mov", ".webm", ".avi", ".mkv")


def load_image(path, max_side=1280):
    """画像を RGB で読む。大きすぎる画像はトークンが増えて遅くなるので縮める (座標は 0〜1000 の比率なので影響しない)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def read_video(path, fps=4, max_side=640, max_frames=256):
    """動画を fps 枚/秒に間引いて読む。返すのはコマの配列と、各コマの時刻を伝えるための情報 (VideoMetadata)。
    Reasoner は各コマに「<1.5 seconds>」のような時刻を付けて読むので、時刻を答える問題 (第4章) にはこの情報が欠かせない"""
    frames, t_next = [], 0.0
    with av.open(path) as c:
        for f in c.decode(video=0):
            if f.time is None or f.time + 1e-6 < t_next:
                continue
            img = f.to_image()
            img.thumbnail((max_side, max_side))  # 1 コマのトークン数を抑える
            frames.append(np.asarray(img.convert("RGB")))
            t_next += 1 / fps
            if len(frames) >= max_frames:
                break
    n = len(frames)
    return np.stack(frames), VideoMetadata(total_num_frames=n, fps=fps, frames_indices=list(range(n)), duration=n / fps)


def ask(prompt, media=None, thinking=False, max_new_tokens=2048, fps=4, seed=0):
    """画像・動画 (なくてもよい) と質問を渡して、答えの文字列を返す。

    thinking=True のときは考えた過程 (<think>…</think>) を出してから答える。考えた過程は ask.last_think に残る。
    動画は fps 枚/秒で間引いて読む (推奨は 4)。
    """
    media = [] if media is None else media if isinstance(media, (list, tuple)) else [media]
    content, metadata = [], []
    for m in media:  # 画像・動画は質問文より前に置く (公式の推奨)
        if isinstance(m, str) and m.lower().endswith(VIDEO_EXT):
            frames, meta = read_video(m, fps=fps)
            content.append({"type": "video", "video": frames})
            metadata.append(meta)
        else:
            content.append({"type": "image", "image": load_image(m)})
    content.append({"type": "text", "text": prompt})
    # 動画は read_video で間引き済みなので、ここではもう間引かない
    video_kwargs = {"video_metadata": metadata, "do_sample_frames": False} if metadata else {}
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        enable_thinking=thinking, **video_kwargs,
    ).to(model.device)
    # 公式のプロンプトガイドのサンプリング設定 (考える / 考えない で値が違う)
    sampling = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    torch.manual_seed(seed)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, **sampling)
    text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ask.last_think = ""
    if "</think>" in text:
        think, text = text.split("</think>", 1)
        ask.last_think = think.replace("<think>", "").strip()
    return text.strip()


def parse_json(text):
    """答えの中の JSON (```json … ``` で囲まれていてもよい) を取り出す。読めなければ None。
    小さなモデルの答えは、最後に余計な括弧が付く・真偽値の後ろに " が付く・最後に余計なカンマがある、
    といった崩れ方をするので、括弧の対応を数えて最初のまとまりだけを取り出し、よくある崩れを直してから読む"""
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    body = m.group(1) if m else text
    starts = [i for i in (body.find("["), body.find("{")) if i >= 0]
    if not starts:
        return None
    start = min(starts)
    depth, in_str, esc, end = 0, False, False, len(body)
    for k in range(start, len(body)):
        ch = body[k]
        if in_str:
            esc = (ch == "\\") and not esc
            if ch == '"' and not esc:
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "[{":
            depth += 1
        elif ch in "]}":
            depth -= 1
            if depth == 0:
                end = k + 1
                break
    chunk = body[start:end]
    fixed = re.sub(r'\b(true|false|null)"', r"\1", re.sub(r",\s*([\]}])", r"\1", chunk))
    for attempt in (chunk, fixed):
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            pass
    print("(JSON として読めませんでした)")
    return None


def load_reasoner():
    """Reasoner を読み込む (生成の章で、Generator と入れ替えて使うとき用)"""
    global model, processor
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model = AutoModelForImageTextToText.from_pretrained("nvidia/Cosmos3-Edge", dtype=torch.bfloat16, device_map="auto")
    processor = AutoProcessor.from_pretrained("nvidia/Cosmos3-Edge")

In [ ]:
# @title 時刻をグラフにする関数・その時刻のコマを切り出す関数
import av
import matplotlib.pyplot as plt


def video_length(path):
    with av.open(path) as c:
        return c.duration / 1e6


def frame_at(path, t):
    """t 秒のコマを PIL 画像で返す"""
    with av.open(path) as c:
        s = c.streams.video[0]
        c.seek(int(max(t, 0) / s.time_base), stream=s)
        last = None
        for f in c.decode(s):
            last = f
            if f.time is not None and f.time >= t:
                break
        return last.to_image()


def to_sec(v):
    """12.3 / "12.3" / "00:12.30" / "0:00:12.3" を秒に直す"""
    if isinstance(v, (int, float)):
        return float(v)
    sec = 0.0
    for part in str(v).strip().split(":"):
        sec = sec * 60 + float(part)
    return sec


def plot_timeline(events, path, title=""):
    """events: [{"start", "end", "caption"}] を 1 行に 1 つずつ横棒で描き、各区間の真ん中のコマを並べる"""
    events = [e for e in events if isinstance(e, dict) and "start" in e and "end" in e]
    n = len(events)
    if n == 0:
        print("区間が見つかりませんでした")
        return
    dur = video_length(path)
    cols = min(n, 5)
    rows = (n + cols - 1) // cols
    fig = plt.figure(figsize=(12, 0.45 * n + 1.2 + 2.0 * rows))
    top = 0.45 * n + 1.2
    ax = fig.add_axes([0.06, 1 - (top - 0.3) / fig.get_figheight(), 0.9, (top - 0.9) / fig.get_figheight()])
    for i, e in enumerate(events):
        s, t = to_sec(e["start"]), to_sec(e["end"])
        ax.barh(i, t - s, left=s, color=COLORS[i % len(COLORS)], edgecolor="k")
        ax.text(s + 0.1, i, str(i + 1), va="center", color="white", weight="bold")
    ax.set_xlim(0, dur)
    ax.set_ylim(n - 0.5, -0.5)
    ax.set_yticks([])
    ax.set_xlabel("seconds")
    ax.set_title(title)
    for i, e in enumerate(events):
        mid = (to_sec(e["start"]) + to_sec(e["end"])) / 2
        sub = fig.add_axes([0.06 + 0.9 * (i % cols) / cols, (2.0 * (rows - 1 - i // cols) + 0.1) / fig.get_figheight(),
                            0.9 / cols - 0.01, 1.7 / fig.get_figheight()])
        sub.imshow(frame_at(path, min(mid, dur - 0.05)))
        sub.set_title(f"{i + 1}: {mid:.1f}s", fontsize=9)
        sub.axis("off")
    plt.show()
    for i, e in enumerate(events):
        print(f"{i + 1}. {to_sec(e['start']):5.1f}s〜{to_sec(e['end']):5.1f}s  {e.get('caption', '')}")


COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#42d4f4", "#f032e6", "#bfef45"]

## 1. 動作を区切って列挙させる

公式のプロンプトガイドのお手本です。ヒューマノイドがポップコーンを入れる動画で、動作の区切りを秒で答えさせます。

In [ ]:
video = fetch(f"{FW}/reasoner/assets/temporal_localization_1.mp4")
print(f"動画の長さ: {video_length(video):.1f} 秒")
answer = ask("""List all action segments in the video.

Provide the result in json format with 'seconds' for time depiction for each event. Use keywords 'start', 'end' and 'caption' in the json output. Please list multiple events if applicable.

```json
[
{
  "start": t_start,
  "end": t_end,
  "caption": EVENT1
},
{
  "start": t_start,
  "end": t_end,
  "caption": EVENT2
},
...
]
```""", video, max_new_tokens=3072)
segments = parse_json(answer) or []
plot_timeline(segments, video, "action segments")

## 2. 出来事の年表を作らせる

時刻の書き方は `mm:ss.ff` (分:秒.百分の一秒) でも頼めます。廊下を人が箱を運んでいく動画です。

In [ ]:
video = fetch(f"{FW}/reasoner/assets/temporal_localization_2.mp4")
print(f"動画の長さ: {video_length(video):.1f} 秒")
answer = ask("Describe the notable events in the provided video. Provide the result in json format with 'mm:ss.ff' format "
             "for time depiction for each event. Use keywords 'start', 'end' and 'caption' in the json output.", video, max_new_tokens=3072)
events = parse_json(answer) or []
plot_timeline(events, video, "event timeline")

## 3. 「この場面はいつ?」と時刻を聞く

探したい場面を文章で書いて、その時刻を答えさせます。長い録画から目当ての場面を探すときに使えます。

In [ ]:
query = "A man in a white sweater walks out of a room carrying a box, closes the door behind him, walks on the floor, and turns left at the end near the wall."
answer = ask(f'When is "{query}" depicted in the video? Please provide the result in json format with \'mm:ss.ff\' format '
             "for time depiction for the event. Use keywords 'start', 'end' in the json output.", video)
print(answer)
found = parse_json(answer)
if isinstance(found, list):
    found = found[0] if found else None
if found:
    found["caption"] = "query"
    plot_timeline([found], video, "timestamp query")

## 4. 「この区間に何が起きた?」と聞く

In [ ]:
print(ask("What happened between 00:05.64 and 00:17.49?", video))

## 5. 車載カメラの動画で試す

モデルカードのお手本の車載カメラの動画 (第9章でも使います) で、運転の場面の区切りを答えさせます。

In [ ]:
video = fetch(f"{HF}/example_action_id_av_0_input.mp4")
display(Video(video, embed=True, width=480))
answer = ask("""This is a video from a forward-facing camera of a car. List the driving events in the video
(for example: driving straight, slowing down, stopping, turning, changing lanes, another vehicle cutting in).
Provide the result in json format with 'seconds' for time depiction for each event.
Use keywords 'start', 'end' and 'caption' in the json output.""", video, max_new_tokens=2048)
plot_timeline(parse_json(answer) or [], video, "driving events")

## 6. 自分の動画で試す

ロボットの作業の録画 (LeRobot で撮ったエピソードなど) を入れると、作業の区切りを自動で付けられます。
データにラベルを付ける作業の下書きに使えます。

In [ ]:
# @title 自分の動画で試す (colab-only)
from google.colab import files

question = "List all action segments in the video. Provide the result in json format with 'seconds' for time depiction for each event. Use keywords 'start', 'end' and 'caption' in the json output."  # @param {type:"string"}

up = files.upload()
path = next(iter(up))
answer = ask(question, path, max_new_tokens=3072)
plot_timeline(parse_json(answer) or [], path, "my video")

## うまくいかないとき

- **時刻が動画の長さを超える・ずれる**: 動画は `fps` (既定 4) で間引いて読んでいるので、細かい時刻は 0.5 秒ほどずれます。速い動作を見たいときは `ask(..., fps=8)` にします (その分重くなります)
- **区間が 1 つしか返ってこない**: 「Please list multiple events if applicable.」を付け、お手本の JSON をそのまま入れると改善します
- **長い動画で GPU メモリが足りない**: 動画を 30 秒程度に切ってから入れるか、`fps=2` にします